# Negative-Peaked Template Examples

Find templates in the default matched-filter library whose nominal peak sample is below the local continuum, then plot a reproducible sample with the parameters used to generate each model.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "searchlight").exists():
            return path
    raise RuntimeError("Could not find the astrosearchlight project root")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from searchlight.matched_filtering import (  # noqa: E402
    DEFAULT_TEMPLATE_LIBRARY_PATH,
    load_template_library,
)

template_library_path = DEFAULT_TEMPLATE_LIBRARY_PATH
template_library_path

If the default file is missing, generate it from the project root with:

```shell
uv sync --extra template-generation --all-groups
uv run python bin/generate_matched_filter_template_library.py
```

In [ ]:
if not template_library_path.exists():
    raise FileNotFoundError(
        f"Template library not found at {template_library_path}. "
        "Generate it with bin/generate_matched_filter_template_library.py."
    )

library = load_template_library(template_library_path)
print(f"Loaded {len(library):,} templates from {template_library_path}")
print(f"Flux array shape: {library.flux.shape}")
print(f"Time range: {library.time[0]:.3f} to {library.time[-1]:.3f} days")
library.metadata

In [ ]:
if library.parameters is None:
    raise ValueError("This template library does not include model parameters.")

parameter_names = library.parameters.dtype.names
print(f"Parameter columns: {', '.join(parameter_names)}")

The template time coordinate is measured relative to the intended peak at `t=0`. For each row, estimate the continuum from the outer 10% of samples, then compare the `t=0` sample to that baseline. The table also records the largest absolute excursion from the same baseline and whether that dominant excursion is negative.

In [ ]:
edge_fraction = 0.10
edge_count = max(1, int(round(edge_fraction * library.flux.shape[1])))
edge_flux = np.concatenate(
    [library.flux[:, :edge_count], library.flux[:, -edge_count:]],
    axis=1,
)
baseline_flux = np.nanmedian(edge_flux, axis=1)
delta_flux = library.flux - baseline_flux[:, np.newaxis]

center_index = int(np.argmin(np.abs(library.time)))
center_flux = library.flux[:, center_index]
center_delta_flux = delta_flux[:, center_index]
dominant_indices = np.argmax(np.abs(delta_flux), axis=1)
dominant_time = library.time[dominant_indices]
dominant_delta_flux = delta_flux[np.arange(len(library)), dominant_indices]

negative_center_mask = center_delta_flux < 0
negative_dominant_mask = dominant_delta_flux < 0
negative_peaked_mask = negative_center_mask
negative_indices = np.flatnonzero(negative_peaked_mask)

def example_record(index):
    record = {
        "template_index": int(index),
        "baseline_flux": float(baseline_flux[index]),
        "center_time": float(library.time[center_index]),
        "center_flux": float(center_flux[index]),
        "center_delta_flux": float(center_delta_flux[index]),
        "dominant_time": float(dominant_time[index]),
        "dominant_delta_flux": float(dominant_delta_flux[index]),
        "negative_center_peak": bool(negative_center_mask[index]),
        "negative_dominant_excursion": bool(negative_dominant_mask[index]),
    }
    record.update(
        {name: float(library.parameters[index][name]) for name in parameter_names}
    )
    return record


negative_examples = [example_record(index) for index in negative_indices]
print(f"Negative center-peak templates: {len(negative_examples):,} / {len(library):,}")
print(
    "Negative dominant excursions among them: "
    f"{sum(record['negative_dominant_excursion'] for record in negative_examples):,}"
)
display(sorted(negative_examples, key=lambda record: record["center_delta_flux"])[:20])

In [ ]:
sample_size = min(12, len(negative_indices))
random_seed = 20260810
rng = np.random.default_rng(random_seed)

if sample_size == 0:
    sample_indices = np.array([], dtype=int)
else:
    sample_indices = rng.choice(
        negative_indices,
        size=sample_size,
        replace=False,
    )

sample = [example_record(index) for index in sample_indices]
display(sample)

In [ ]:
def parameter_label(row):
    return (
        f"#{row['template_index']} | a={row['a']:g} AU, e={row['e']:g}, "
        f"phase={row['periapsis_phase']:.3g} rad\n"
        f"Macc={row['Macc']:g} Msun, Mcomp={row['Mcomp']:g} Msun, "
        f"b={row['b']:.3g}, tau={row['tau_eff']:.3g} d"
    )


if not sample:
    print("No negative-peaked examples found with the current criterion.")
else:
    cols = min(3, len(sample))
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 3.4 * rows), squeeze=False)

    for ax, row in zip(axes.ravel(), sample):
        index = row["template_index"]
        ax.plot(library.time, library.flux[index], color="tab:blue", linewidth=1.4)
        ax.axhline(row["baseline_flux"], color="0.55", linestyle="--", linewidth=1)
        ax.axvline(0, color="0.75", linewidth=1)
        ax.scatter(
            [row["center_time"]],
            [row["center_flux"]],
            color="tab:red",
            s=28,
            zorder=3,
            label="t=0 sample",
        )
        ax.scatter(
            [row["dominant_time"]],
            [row["baseline_flux"] + row["dominant_delta_flux"]],
            color="tab:orange",
            marker="x",
            s=42,
            zorder=3,
            label="dominant excursion",
        )
        ax.set_title(parameter_label(row), fontsize=9)
        ax.set_xlabel("Days from nominal peak")
        ax.set_ylabel("Relative flux")
        ax.legend(loc="best", fontsize=8)

    for ax in axes.ravel()[len(sample):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()

To inspect only templates where the largest absolute excursion is also negative, replace `negative_peaked_mask = negative_center_mask` above with `negative_peaked_mask = negative_center_mask & negative_dominant_mask` and rerun the notebook.